# OpenDataBattle2026 Phase2 データ加工【盆地】

#### [1]読み込み
#### [2]前処理
#### [3]GISと標高タイルを用いた特徴量計算

#### 1.各県庁所在地のある市域の標高を読み込み以下を算出
- 最大標高と最小標高の差
- 市域の標高分布の標準偏差（どのくらい散らばっているか）
#### 2.各県庁舎からみて360度を1度刻みにし、半径〇km以内の仰角について以下を算出
##### (半径は3パターン、5km,10km,20km)
- 閉塞方向率（仰角が0度以上の方向は"閉じている"と定義。仰角が算出できた方角のうち閉じている方角の割合）
- 平均最大仰角（1度につき最大の仰角を算出。算出した各法学の最大仰角の平均値）
- 最大仰角（仰角を算出できた方角のうち、最大の仰角）

## 【読み込み】

### ライブラリ読み込み

In [1]:
import io
import math
import requests
import numpy as np
import pandas as pd
import geopandas as gpd

from shapely.geometry import Point

from PIL import Image
from pyproj import Transformer

### データ読み込み

In [2]:
# 県庁所在地一覧
df_prefectural_capitals = pd.read_csv("../data/raw_00_reference_県庁所在地一覧_20260803.csv", encoding='shift_jis')
# 各庁舎の緯度経度
df_capital_location = pd.read_csv("../data/data_capital_location.csv", index_col=0)

# 行政区域データ → Python3.12環境で作業
gdf = gpd.read_file("../data/GIS_行政区域データ_20260803/N03-23_230101.shp", encoding="cp932")

# 日本百名山データ
#df_hyakumeizan = pd.read_csv("../data/raw_03_日本百名山_20260901.csv", encoding="cp932")

## 【データ確認】

### 県庁所在地一覧データ

In [3]:
# 末尾３件確認
df_prefectural_capitals.tail(3)

,コード,都道府県,県庁所在地,ローマ字
44,45,宮崎県,宮崎市,miyazaki
45,46,鹿児島県,鹿児島市,kagoshima
46,47,沖縄県,那覇市,naha


In [4]:
# 概要
df_prefectural_capitals.info()

<class 'pandas.DataFrame'>
RangeIndex: 47 entries, 0 to 46
Data columns (total 4 columns):
 #   Column  Non-Null Count  Dtype
---  ------  --------------  -----
 0   コード     47 non-null     int64
 1   都道府県    47 non-null     str  
 2   県庁所在地   47 non-null     str  
 3   ローマ字    47 non-null     str  
dtypes: int64(1), str(3)
memory usage: 1.6 KB


### 各庁舎の緯度経度

In [5]:
df_capital_location.head(3)

,都道府県,市区,緯度,経度,都市
0,北海道,札幌市,43.064310,141.346879,札幌
1,青森県,青森市,40.824589,140.740548,青森
2,岩手県,盛岡市,39.703526,141.152696,盛岡


In [6]:
df_capital_location.info()

<class 'pandas.DataFrame'>
RangeIndex: 47 entries, 0 to 46
Data columns (total 5 columns):
 #   Column  Non-Null Count  Dtype  
---  ------  --------------  -----  
 0   都道府県    47 non-null     str    
 1   市区      47 non-null     str    
 2   緯度      47 non-null     float64
 3   経度      47 non-null     float64
 4   都市      47 non-null     str    
dtypes: float64(2), str(3)
memory usage: 2.0 KB


## 行政区域データ(GIS)

In [7]:
gdf.head(3)

,OBJECTID,N03_001,N03_002,N03_003,N03_004,N03_007,Shape_Leng,Shape_Area,geometry
0,1.0,北海道,石狩振興局,札幌市,札幌市中央区,01101,0.551644,0.005128,"POLYGON ((141.34233 43.06682, 141.34285 43.066..."
1,2.0,北海道,石狩振興局,札幌市,札幌市北区,01102,0.579442,0.007033,"POLYGON ((141.40839 43.18395, 141.40833 43.183..."
2,3.0,北海道,石狩振興局,札幌市,札幌市東区,01103,0.401459,0.006300,"POLYGON ((141.44707 43.15616, 141.44694 43.155..."


In [8]:
gdf.info()

<class 'geopandas.geodataframe.GeoDataFrame'>
RangeIndex: 122929 entries, 0 to 122928
Data columns (total 9 columns):
 #   Column      Non-Null Count   Dtype   
---  ------      --------------   -----   
 0   OBJECTID    122929 non-null  float64 
 1   N03_001     122929 non-null  str     
 2   N03_002     9575 non-null    str     
 3   N03_003     51485 non-null   str     
 4   N03_004     122929 non-null  str     
 5   N03_007     122929 non-null  str     
 6   Shape_Leng  122929 non-null  float64 
 7   Shape_Area  122929 non-null  float64 
 8   geometry    122929 non-null  geometry
dtypes: float64(3), geometry(1), str(5)
memory usage: 8.4 MB


### 日本百名山データ

In [11]:
# 末尾３件確認
#df_hyakumeizan.head(3)

In [10]:
# 概要
#df_hyakumeizan.info()

## 【前処理】

### 県庁所在地データ

In [12]:
### 東京都の県庁所在地が「東京都」になっているので「東京」に変更
df_prefectural_capitals["県庁所在地"] = df_prefectural_capitals["県庁所在地"].replace(
    "東京都", "東京"
)
df_prefectural_capitals["県庁所在地"].unique()

<StringArray>
[  '札幌市',   '青森市',   '盛岡市',   '仙台市',   '秋田市',   '山形市',   '福島市',   '水戸市',
  '宇都宮市',   '前橋市', 'さいたま市',   '千葉市',    '東京',   '横浜市',   '新潟市',   '富山市',
   '金沢市',   '福井市',   '甲府市',   '長野市',   '岐阜市',   '静岡市',  '名古屋市',    '津市',
   '大津市',   '京都市',   '大阪市',   '神戸市',   '奈良市',  '和歌山市',   '鳥取市',   '松江市',
   '岡山市',   '広島市',   '山口市',   '徳島市',   '高松市',   '松山市',   '高知市',   '福岡市',
   '佐賀市',   '長崎市',   '熊本市',   '大分市',   '宮崎市',  '鹿児島市',   '那覇市']
Length: 47, dtype: str

In [13]:
# 結合キー用に都市名から"市"を抜いた列を作成
### 末尾についた"市"だけ削除removesuffix()
df_prefectural_capitals["都市"] = (
    df_prefectural_capitals["県庁所在地"]
    .str.removesuffix("市")
)
df_prefectural_capitals.head()

,コード,都道府県,県庁所在地,ローマ字,都市
0,1,北海道,札幌市,sapporo,札幌
1,2,青森県,青森市,aomori,青森
2,3,岩手県,盛岡市,morioka,盛岡
3,4,宮城県,仙台市,sendai,仙台
4,5,秋田県,秋田市,akita,秋田


### 県庁の緯度経度

In [14]:
# GIS用にPoint化
gdf_capital_points = gpd.GeoDataFrame(
    df_capital_location.copy(),
    geometry=[
        Point(lon, lat)
        for lon, lat in zip(
            df_capital_location["経度"],
            df_capital_location["緯度"]
        )
    ],
    crs="EPSG:4326"
)

## 【各県庁所在地のある市内の標高について】

### タイル取得関数

In [15]:
# 経度・緯度 → XYZタイル番号
def lonlat_to_tile(lon, lat, z):
    """
    緯度経度 → XYZタイル番号
    """
    n = 2 ** z

    x = int((lon + 180) / 360 * n)

    lat_rad = math.radians(lat)
    y = int(
        (1 - math.asinh(math.tan(lat_rad)) / math.pi) / 2 * n
    )

    return x, y

In [16]:
# PNGタイルの取得
def get_dem_tile(z, x, y):
    url = f"https://cyberjapandata.gsi.go.jp/xyz/dem_png/{z}/{x}/{y}.png"

    response = requests.get(url)
    response.raise_for_status()

    return np.array(Image.open(io.BytesIO(response.content)).convert("RGB"))

In [17]:
# RGB → 標高
def rgb_to_elevation(rgb):
    r = rgb[:, :, 0].astype(np.int64)
    g = rgb[:, :, 1].astype(np.int64)
    b = rgb[:, :, 2].astype(np.int64)

    x = 2**16 * r + 2**8 * g + b

    elevation = np.where(
        x < 2**23,
        x * 0.01,
        (x - 2**24) * 0.01
    )

    # NoData
    elevation[(r == 128) & (g == 0) & (b == 0)] = np.nan

    # x == 2^23 も欠測
    elevation[x == 2**23] = np.nan

    return elevation

### 都市検索用に一覧を作成

In [23]:
# 東京を除いた県庁所在地
capital_cities = df_prefectural_capitals.loc[
    df_prefectural_capitals["県庁所在地"] != "東京",
    "県庁所在地"
].tolist()

# 一般市：N03_004 に県庁所在地が入っているもの
gdf_general = gdf[
    gdf["N03_004"].isin(capital_cities)
].copy()

# 政令指定都市：N03_003 に県庁所在地が入っているもの
gdf_designated = gdf[
    gdf["N03_003"].isin(capital_cities)
].copy()

# 結合
gdf_capitals = pd.concat(
    [gdf_general, gdf_designated],
    ignore_index=True
)

print(gdf_capitals.shape)
print(gdf_capitals["N03_003"].dropna().unique())

(6986, 9)
<StringArray>
[  '札幌市',   '仙台市', 'さいたま市',   '千葉市',   '横浜市',   '新潟市',   '静岡市',  '名古屋市',
   '京都市',   '大阪市',   '神戸市',   '岡山市',   '広島市',   '福岡市',   '熊本市']
Length: 15, dtype: str


In [29]:
# 県庁所在地名を格納する列を作成
gdf_capitals["県庁所在地"] = gdf_capitals["N03_004"]

# 政令指定都市については N03_003 を使用
mask = gdf_capitals["N03_003"].isin(capital_cities)

gdf_capitals.loc[mask, "県庁所在地"] = (
    gdf_capitals.loc[mask, "N03_003"]
)

In [30]:
gdf_capitals_city = (
    gdf_capitals
    .dissolve(by="県庁所在地")
    .reset_index()
)
print(gdf_capitals_city.shape)
print(gdf_capitals_city["県庁所在地"].tolist())

(46, 10)
['さいたま市', '京都市', '仙台市', '佐賀市', '前橋市', '千葉市', '名古屋市', '和歌山市', '大分市', '大津市', '大阪市', '奈良市', '宇都宮市', '宮崎市', '富山市', '山口市', '山形市', '岐阜市', '岡山市', '広島市', '徳島市', '新潟市', '札幌市', '松山市', '松江市', '横浜市', '水戸市', '津市', '熊本市', '甲府市', '盛岡市', '神戸市', '福井市', '福岡市', '福島市', '秋田市', '那覇市', '金沢市', '長崎市', '長野市', '青森市', '静岡市', '高松市', '高知市', '鳥取市', '鹿児島市']


In [31]:
gdf_capitals_city[
    gdf_capitals_city["県庁所在地"].isin(["盛岡市", "福岡市"])
][["県庁所在地", "geometry"]]

,県庁所在地,geometry
30,盛岡市,"POLYGON ((141.16668 39.92986, 141.16669 39.929..."
33,福岡市,"MULTIPOLYGON (((130.20015 33.64424, 130.20014 ..."


In [33]:
# 23区それぞれの行政区域を dissolve() して一つの東京23区ポリゴンにする

tokyo_23ku = [
    "千代田区", "中央区", "港区", "新宿区", "文京区",
    "台東区", "墨田区", "江東区", "品川区", "目黒区",
    "大田区", "世田谷区", "渋谷区", "中野区", "杉並区",
    "豊島区", "北区", "荒川区", "板橋区", "練馬区",
    "足立区", "葛飾区", "江戸川区"
]

gdf_tokyo = gdf[
    gdf["N03_004"].isin(tokyo_23ku)
].copy()

gdf_tokyo["県庁所在地"] = "東京"

gdf_tokyo_city = (
    gdf_tokyo
    .dissolve(by="県庁所在地")
    .reset_index()
)

print(gdf_tokyo_city.shape)
print(gdf_tokyo_city[["県庁所在地", "geometry"]])

(1, 10)
  県庁所在地                                           geometry
0    東京  MULTIPOLYGON (((139.75313 35.56271, 139.75331 ...


In [34]:
# 47県庁所在地になっていることを確認
gdf_capitals_city = pd.concat(
    [gdf_capitals_city, gdf_tokyo_city],
    ignore_index=True
)
print(gdf_capitals_city.shape)
print(gdf_capitals_city["県庁所在地"].nunique())
print(sorted(gdf_capitals_city["県庁所在地"].tolist()))

(47, 10)
47
['さいたま市', '京都市', '仙台市', '佐賀市', '前橋市', '千葉市', '名古屋市', '和歌山市', '大分市', '大津市', '大阪市', '奈良市', '宇都宮市', '宮崎市', '富山市', '山口市', '山形市', '岐阜市', '岡山市', '広島市', '徳島市', '新潟市', '札幌市', '東京', '松山市', '松江市', '横浜市', '水戸市', '津市', '熊本市', '甲府市', '盛岡市', '神戸市', '福井市', '福岡市', '福島市', '秋田市', '那覇市', '金沢市', '長崎市', '長野市', '青森市', '静岡市', '高松市', '高知市', '鳥取市', '鹿児島市']


In [35]:
# 47都市のジオメトリが正常かだけ確認
print(gdf_capitals_city.crs)
print(gdf_capitals_city.geometry.is_valid.value_counts())
print(gdf_capitals_city.geometry.geom_type.value_counts())

EPSG:6668
True    47
Name: count, dtype: int64
MultiPolygon    32
Polygon         15
Name: count, dtype: int64


In [36]:
# DEMタイルのXYZは通常、経度緯度（WGS84）を前提にタイル番号を計算するので、処理用に一時的に EPSG:4326 に変換
gdf_capitals_dem = gdf_capitals_city.to_crs("EPSG:4326").copy()
print(gdf_capitals_dem.crs)
print(gdf_capitals_dem.shape)

EPSG:4326
(47, 10)


### （盛岡でおためし）

In [37]:
# 盛岡市のbounding boxを確認

city = "盛岡市"

geom = gdf_capitals_dem.loc[
    gdf_capitals_dem["県庁所在地"] == city,
    "geometry"
].iloc[0]

min_lon, min_lat, max_lon, max_lat = geom.bounds

print("bbox:")
print(min_lon, min_lat, max_lon, max_lat)

bbox:
140.99526697848842 39.563732721587996 141.52747797722702 39.930181865201064


In [38]:
# z=14で必要なタイル範囲を求める

z = 14

x_min, y_max = lonlat_to_tile(min_lon, min_lat, z)
x_max, y_min = lonlat_to_tile(max_lon, max_lat, z)

print("x:", x_min, "～", x_max)
print("y:", y_min, "～", y_max)

print(
    "必要タイル数:",
    (x_max - x_min + 1) * (y_max - y_min + 1)
)

x: 14608 ～ 14633
y: 6206 ～ 6228
必要タイル数: 598


In [39]:
# まず1枚だけ取得して、標高配列を確認

x = x_min
y = y_min

rgb = get_dem_tile(z, x, y)
elevation = rgb_to_elevation(rgb)

print(rgb.shape)
print(elevation.shape)
print(np.nanmin(elevation))
print(np.nanmax(elevation))

(256, 256, 3)
(256, 256)
376.1
634.24


In [40]:
# タイル内の各ピクセルの緯度経度を求める

def tile_pixels_to_lonlat(z, x, y, width=256, height=256):
    n = 2 ** z

    cols = np.arange(width) + 0.5
    rows = np.arange(height) + 0.5

    xx, yy = np.meshgrid(cols, rows)

    tile_x = x + xx / width
    tile_y = y + yy / height

    lon = tile_x / n * 360.0 - 180.0

    lat = np.degrees(
        np.arctan(
            np.sinh(
                np.pi * (1 - 2 * tile_y / n)
            )
        )
    )

    return lon, lat

In [41]:
lon, lat = tile_pixels_to_lonlat(z, x, y)

print(lon.shape)
print(lat.shape)

print("経度:", lon.min(), "～", lon.max())
print("緯度:", lat.min(), "～", lat.max())

(256, 256)
(256, 256)
経度: 140.97660541534424 ～ 140.99849224090576
緯度: 39.92662133228002 ～ 39.943403559688434


In [42]:
# 盛岡市ポリゴン内のピクセルだけを判定

from shapely import contains_xy

mask = contains_xy(geom, lon, lat)

print(mask.shape)
print("市域内ピクセル数:", mask.sum())
print("全ピクセル数:", mask.size)

(256, 256)
市域内ピクセル数: 0
全ピクセル数: 65536


In [45]:
tiles = []

for x in range(x_min, x_max + 1):
    for y in range(y_min, y_max + 1):
        tiles.append((x, y))

print("必要タイル数:", len(tiles))
print(tiles)

必要タイル数: 598
[(14608, 6206), (14608, 6207), (14608, 6208), (14608, 6209), (14608, 6210), (14608, 6211), (14608, 6212), (14608, 6213), (14608, 6214), (14608, 6215), (14608, 6216), (14608, 6217), (14608, 6218), (14608, 6219), (14608, 6220), (14608, 6221), (14608, 6222), (14608, 6223), (14608, 6224), (14608, 6225), (14608, 6226), (14608, 6227), (14608, 6228), (14609, 6206), (14609, 6207), (14609, 6208), (14609, 6209), (14609, 6210), (14609, 6211), (14609, 6212), (14609, 6213), (14609, 6214), (14609, 6215), (14609, 6216), (14609, 6217), (14609, 6218), (14609, 6219), (14609, 6220), (14609, 6221), (14609, 6222), (14609, 6223), (14609, 6224), (14609, 6225), (14609, 6226), (14609, 6227), (14609, 6228), (14610, 6206), (14610, 6207), (14610, 6208), (14610, 6209), (14610, 6210), (14610, 6211), (14610, 6212), (14610, 6213), (14610, 6214), (14610, 6215), (14610, 6216), (14610, 6217), (14610, 6218), (14610, 6219), (14610, 6220), (14610, 6221), (14610, 6222), (14610, 6223), (14610, 6224), (14610, 6225

In [46]:
city_elevations = []

for x, y in tiles:
    rgb = get_dem_tile(z, x, y)
    elevation = rgb_to_elevation(rgb)

    lon, lat = tile_pixels_to_lonlat(z, x, y)

    mask = contains_xy(geom, lon, lat)

    values = elevation[mask]
    values = values[~np.isnan(values)]

    if len(values) > 0:
        city_elevations.append(values)

city_elevations = np.concatenate(city_elevations)

print("市域内DEM数:", len(city_elevations))
print("最低標高:", np.min(city_elevations))
print("最高標高:", np.max(city_elevations))
print("標高差:", np.max(city_elevations) - np.min(city_elevations))

市域内DEM数: 16445351
最低標高: 97.5
最高標高: 1424.0
標高差: 1326.5


### 47都市分の計算（標高差）

In [47]:
# 1. DEMキャッシュ用フォルダ

from pathlib import Path

dem_cache_dir = Path("../data/DEM10B_cache_z14")
dem_cache_dir.mkdir(parents=True, exist_ok=True)

In [51]:
# 2. キャッシュ対応のDEM取得関数

def get_dem_tile(z, x, y):
    cache_path = dem_cache_dir / f"{z}_{x}_{y}.png"

    if cache_path.exists():
        image = Image.open(cache_path).convert("RGB")
        return np.array(image)

    url = f"https://cyberjapandata.gsi.go.jp/xyz/dem_png/{z}/{x}/{y}.png"

    response = requests.get(url, timeout=30)

    # DEMタイルが存在しない場合
    if response.status_code == 404:
        return None

    response.raise_for_status()

    cache_path.write_bytes(response.content)

    image = Image.open(io.BytesIO(response.content)).convert("RGB")
    return np.array(image)

In [55]:
def get_dem_tile(z, x, y, max_retries=3):
    cache_path = dem_cache_dir / f"{z}_{x}_{y}.png"

    # キャッシュがあれば利用
    if cache_path.exists():
        image = Image.open(cache_path).convert("RGB")
        return np.array(image)

    url = f"https://cyberjapandata.gsi.go.jp/xyz/dem_png/{z}/{x}/{y}.png"

    for attempt in range(1, max_retries + 1):

        try:
            response = requests.get(
                url,
                timeout=30
            )

            # タイルが存在しない
            if response.status_code == 404:
                return None

            response.raise_for_status()

            # 正常取得したら保存
            cache_path.write_bytes(response.content)

            image = Image.open(
                io.BytesIO(response.content)
            ).convert("RGB")

            return np.array(image)

        except requests.exceptions.RequestException as e:

            print(
                f"\n    通信エラー "
                f"(attempt {attempt}/{max_retries}): "
                f"{type(e).__name__}"
            )

            if attempt < max_retries:
                import time
                time.sleep(2)

    # 3回とも失敗した場合
    print(
        f"\n    タイル取得失敗: "
        f"z={z}, x={x}, y={y}"
    )

    return None

In [56]:
# 3. 47都市を処理

z = 14

elevation_results = []

total_cities = len(gdf_capitals_dem)

for city_idx, row in enumerate(
    gdf_capitals_dem.itertuples(index=False),
    start=1
):
    city = row.県庁所在地
    geom = row.geometry

    min_lon, min_lat, max_lon, max_lat = geom.bounds

    x_min, y_max = lonlat_to_tile(min_lon, min_lat, z)
    x_max, y_min = lonlat_to_tile(max_lon, max_lat, z)

    tiles = [
        (x, y)
        for x in range(x_min, x_max + 1)
        for y in range(y_min, y_max + 1)
    ]

    city_min = np.inf
    city_max = -np.inf

    print(
        f"\n[{city_idx}/{total_cities}] {city} "
        f"({len(tiles)} tiles)"
    )

    for tile_idx, (x, y) in enumerate(tiles, start=1):

        print(
            f"  tile {tile_idx}/{len(tiles)} "
            f"(x={x}, y={y})",
            end="\r"
        )

        rgb = get_dem_tile(z, x, y)

        # タイルが存在しない場合はスキップ
        if rgb is None:
            continue

        elevation = rgb_to_elevation(rgb)

        lon, lat = tile_pixels_to_lonlat(z, x, y)

        mask = contains_xy(geom, lon, lat)

        values = elevation[mask]
        values = values[~np.isnan(values)]

        if len(values) > 0:
            city_min = min(city_min, values.min())
            city_max = max(city_max, values.max())

    if np.isfinite(city_min) and np.isfinite(city_max):

        elevation_range = city_max - city_min

        elevation_results.append({
            "県庁所在地": city,
            "最低標高_m": city_min,
            "最高標高_m": city_max,
            "市域内標高差_m": elevation_range
        })

        print(
            f"  → 完了: "
            f"min={city_min:.1f} m, "
            f"max={city_max:.1f} m, "
            f"差={elevation_range:.1f} m"
        )

    else:
        print("  → DEM値を取得できませんでした")


[1/47] さいたま市 (121 tiles)
  → 完了: min=0.0 m, max=20.7 m, 差=20.7 m

[2/47] 京都市 (390 tiles)
  → 完了: min=5.9 m, max=970.3 m, 差=964.4 m

[3/47] 仙台市 (493 tiles)
  → 完了: min=0.0 m, max=1491.9 m, 差=1491.9 m

[4/47] 佐賀市 (228 tiles)
  → 完了: min=0.0 m, max=1041.9 m, 差=1041.9 m

[5/47] 前橋市 (154 tiles)
  → 完了: min=65.0 m, max=1820.0 m, 差=1755.0 m

[6/47] 千葉市 (196 tiles)
  → 完了: min=0.0 m, max=104.6 m, 差=104.6 m

[7/47] 名古屋市 (169 tiles)
  → 完了: min=-4.6 m, max=196.3 m, 差=200.8 m

[8/47] 和歌山市 (160 tiles)
  → 完了: min=0.0 m, max=483.6 m, 差=483.6 m

[9/47] 大分市 (325 tiles)
  → 完了: min=0.0 m, max=853.5 m, 差=853.5 m

[10/47] 大津市 (264 tiles)
  → 完了: min=71.7 m, max=1212.1 m, 差=1140.4 m

[11/47] 大阪市 (143 tiles)
  → 完了: min=-1.9 m, max=37.3 m, 差=39.2 m

[12/47] 奈良市 (204 tiles)
  → 完了: min=40.3 m, max=820.4 m, 差=780.2 m

[13/47] 宇都宮市 (224 tiles)
  → 完了: min=71.2 m, max=581.5 m, 差=510.3 m

[14/47] 宮崎市 (320 tiles)
  → 完了: min=0.0 m, max=1116.6 m, 差=1116.6 m

[15/47] 富山市 (736 tiles)
  → 完了: min=0.0 m, max=2978.8

In [57]:
# 4. 結果をDataFrame化

df_elevation = pd.DataFrame(elevation_results)

print(df_elevation.shape)
print(df_elevation)

(47, 4)
    県庁所在地  最低標高_m   最高標高_m  市域内標高差_m
0   さいたま市    0.00    20.70     20.70
1     京都市    5.91   970.28    964.37
2     仙台市    0.00  1491.89   1491.89
3     佐賀市    0.00  1041.88   1041.88
4     前橋市   65.03  1819.99   1754.96
5     千葉市    0.00   104.61    104.61
6    名古屋市   -4.56   196.26    200.82
7    和歌山市    0.00   483.64    483.64
8     大分市    0.00   853.48    853.48
9     大津市   71.67  1212.09   1140.42
10    大阪市   -1.90    37.31     39.21
11    奈良市   40.27   820.43    780.16
12   宇都宮市   71.20   581.51    510.31
13    宮崎市    0.00  1116.59   1116.59
14    富山市    0.00  2978.84   2978.84
15    山口市    0.00  1040.34   1040.34
16    山形市   85.11  1840.64   1755.53
17    岐阜市    5.73   415.50    409.77
18    岡山市   -3.34   555.09    558.43
19    広島市    0.00  1041.79   1041.79
20    徳島市    0.00   771.46    771.46
21    新潟市   -5.00   632.45    637.45
22    札幌市    1.00  1486.63   1485.63
23    松山市    0.00  1214.37   1214.37
24    松江市    0.00   609.62    609.62
25    横浜市    0.00   156.96    

### 47都市分の計算（市内標高の標準偏差）

In [59]:
# 市域内標高の標準偏差を計算

z = 14

elevation_std_results = []
total_cities = len(gdf_capitals_dem)

for city_idx, row in enumerate(
    gdf_capitals_dem.itertuples(index=False),
    start=1
):
    city = row.県庁所在地
    geom = row.geometry

    min_lon, min_lat, max_lon, max_lat = geom.bounds

    x_min, y_max = lonlat_to_tile(min_lon, min_lat, z)
    x_max, y_min = lonlat_to_tile(max_lon, max_lat, z)

    tiles = [
        (x, y)
        for x in range(x_min, x_max + 1)
        for y in range(y_min, y_max + 1)
    ]

    # 標高の集計値
    n = 0
    sum_elevation = 0.0
    sum_elevation_sq = 0.0

    print(f"\n[{city_idx}/{total_cities}] {city} ({len(tiles)} tiles)")

    for tile_idx, (x, y) in enumerate(tiles, start=1):
        print(
            f"  tile {tile_idx}/{len(tiles)} (x={x}, y={y})",
            end="\r"
        )

        rgb = get_dem_tile(z, x, y)

        if rgb is None:
            continue

        elevation = rgb_to_elevation(rgb)

        lon, lat = tile_pixels_to_lonlat(z, x, y)
        mask = contains_xy(geom, lon, lat)

        values = elevation[mask]
        values = values[~np.isnan(values)]

        if len(values) > 0:
            n += len(values)
            sum_elevation += np.sum(values)
            sum_elevation_sq += np.sum(values ** 2)

    if n > 0:
        mean_elevation = sum_elevation / n

        # 母標準偏差
        variance = (sum_elevation_sq / n) - (mean_elevation ** 2)

        # 浮動小数点誤差によるごく小さな負値を防ぐ
        variance = max(variance, 0)

        std_elevation = np.sqrt(variance)

        elevation_std_results.append({
            "県庁所在地": city,
            "市域内平均標高_m": mean_elevation,
            "市域内標高標準偏差_m": std_elevation
        })

        print(
            f"  → 完了: mean={mean_elevation:.1f} m, "
            f"std={std_elevation:.1f} m"
        )
    else:
        print("  → DEM値を取得できませんでした")


[1/47] さいたま市 (121 tiles)
  → 完了: mean=9.9 m, std=3.0 m43)

[2/47] 京都市 (390 tiles)
  → 完了: mean=367.6 m, std=234.3 m

[3/47] 仙台市 (493 tiles)
  → 完了: mean=302.0 m, std=298.2 m

[4/47] 佐賀市 (228 tiles)
  → 完了: mean=229.8 m, std=245.1 m

[5/47] 前橋市 (154 tiles)
  → 完了: mean=338.2 m, std=350.0 m

[6/47] 千葉市 (196 tiles)
  → 完了: mean=30.2 m, std=22.1 m)

[7/47] 名古屋市 (169 tiles)
  → 完了: mean=17.8 m, std=21.4 m)

[8/47] 和歌山市 (160 tiles)
  → 完了: mean=56.0 m, std=81.0 m)

[9/47] 大分市 (325 tiles)
  → 完了: mean=140.6 m, std=157.1 m

[10/47] 大津市 (264 tiles)
  → 完了: mean=279.3 m, std=239.7 m

[11/47] 大阪市 (143 tiles)
  → 完了: mean=3.5 m, std=3.7 m12)

[12/47] 奈良市 (204 tiles)
  → 完了: mean=286.3 m, std=164.7 m

[13/47] 宇都宮市 (224 tiles)
  → 完了: mean=159.9 m, std=70.7 m

[14/47] 宮崎市 (320 tiles)
  → 完了: mean=125.9 m, std=145.4 m

[15/47] 富山市 (736 tiles)
  → 完了: mean=801.4 m, std=767.0 m

[16/47] 山口市 (720 tiles)
  → 完了: mean=285.3 m, std=190.3 m

[17/47] 山形市 (221 tiles)
  → 完了: mean=453.7 m, std=348.4 m

[18/47

In [60]:
# 4. 結果をDataFrame化

df_elevation_std = pd.DataFrame(elevation_std_results)

print(df_elevation_std.shape)
print(df_elevation_std)

(47, 3)
    県庁所在地   市域内平均標高_m  市域内標高標準偏差_m
0   さいたま市    9.872245     2.982093
1     京都市  367.637595   234.298340
2     仙台市  301.955290   298.222983
3     佐賀市  229.757853   245.143725
4     前橋市  338.186951   350.039551
5     千葉市   30.179105    22.129316
6    名古屋市   17.848768    21.359561
7    和歌山市   56.034990    80.956537
8     大分市  140.560448   157.128526
9     大津市  279.331607   239.731025
10    大阪市    3.513725     3.747431
11    奈良市  286.308349   164.660501
12   宇都宮市  159.926210    70.727278
13    宮崎市  125.866664   145.428003
14    富山市  801.425753   767.013150
15    山口市  285.276419   190.306043
16    山形市  453.660606   348.359699
17    岐阜市   48.089413    55.636711
18    岡山市   96.381962   106.751522
19    広島市  287.418327   207.804806
20    徳島市   55.685634    99.179345
21    新潟市   12.402606    41.674413
22    札幌市  420.611535   348.341252
23    松山市  223.566023   235.544906
24    松江市   80.213691    96.488838
25    横浜市   35.393915    24.174844
26    水戸市   29.436456    21.894314
27     津市  2

### 標高差と標準偏差を結合

In [76]:
# 標高についての計算結果をマージ
df_elevation = df_elevation.merge(
    df_elevation_std,
    on="県庁所在地",
    how="left"
)
df_elevation.head(2)

,県庁所在地,最低標高_m,最高標高_m,市域内標高差_m,市域内平均標高_m,市域内標高標準偏差_m
0,さいたま市,0.00,20.70,20.70,9.872245,2.982093
1,京都市,5.91,970.28,964.37,367.637595,234.298340


## 【各県庁から見て360度分の仰角について】

In [65]:
def get_dem_tile(z, x, y):
    cache_path = dem_cache_dir / f"{z}_{x}_{y}.png"

    # キャッシュがあれば通信しない
    if cache_path.exists():
        image = Image.open(cache_path).convert("RGB")
        return np.array(image)

    url = (
        f"https://cyberjapandata.gsi.go.jp/"
        f"xyz/dem_png/{z}/{x}/{y}.png"
    )

    try:
        response = requests.get(
            url,
            timeout=(10, 15)
        )

        # 存在しないタイル
        if response.status_code == 404:
            return None

        response.raise_for_status()

        cache_path.write_bytes(
            response.content
        )

        image = Image.open(
            io.BytesIO(response.content)
        ).convert("RGB")

        return np.array(image)

    except requests.exceptions.RequestException as e:

        print(
            f"\n    ⚠ 通信エラー: "
            f"z={z}, x={x}, y={y} "
            f"({type(e).__name__})"
        )

        return None

### 県庁から半径20km

In [66]:
from pyproj import Geod

# ============================================================
# 設定
# ============================================================

z = 14

RADIUS_KM = 20
RADIUS_M = RADIUS_KM * 1000

# 100m間隔
STEP_M = 100

# 1度刻み
AZIMUTHS_DEG = np.arange(0, 360, 1)

# 100m, 200m, ..., 20km
DISTANCES_M = np.arange(
    STEP_M,
    RADIUS_M + 1,
    STEP_M
)

geod = Geod(ellps="WGS84")


# ============================================================
# 緯度経度 → DEMタイル上の座標
# ============================================================

def lonlat_to_tile_float(lon, lat, z):
    n = 2 ** z

    x = (lon + 180.0) / 360.0 * n

    lat_rad = np.radians(lat)

    y = (
        (1.0 - np.arcsinh(np.tan(lat_rad)) / np.pi)
        / 2.0
        * n
    )

    return x, y


# ============================================================
# 指定した緯度経度のDEM標高を取得
# ============================================================

def get_elevation_from_tiles(lons, lats, tiles, z):

    tile_x_float, tile_y_float = lonlat_to_tile_float(
        lons,
        lats,
        z
    )

    tile_x = np.floor(tile_x_float).astype(int)
    tile_y = np.floor(tile_y_float).astype(int)

    pixel_x = (
        (tile_x_float - tile_x) * 256
    ).astype(int)

    pixel_y = (
        (tile_y_float - tile_y) * 256
    ).astype(int)

    pixel_x = np.clip(pixel_x, 0, 255)
    pixel_y = np.clip(pixel_y, 0, 255)

    elevations = np.full(
        len(lons),
        np.nan,
        dtype=float
    )

    unique_tiles = np.unique(
        np.column_stack([tile_x, tile_y]),
        axis=0
    )

    for tx, ty in unique_tiles:

        key = (tx, ty)

        if key not in tiles:
            continue

        mask = (
            (tile_x == tx)
            & (tile_y == ty)
        )

        elevations[mask] = tiles[key][
            pixel_y[mask],
            pixel_x[mask]
        ]

    return elevations


# ============================================================
# 20km圏内をカバーするDEMタイルを取得
# ============================================================

def load_dem_tiles_for_radius(
    lon,
    lat,
    radius_m,
    z
):

    # 東西南北の端点
    lon_n, lat_n, _ = geod.fwd(
        lon, lat, 0, radius_m
    )

    lon_e, lat_e, _ = geod.fwd(
        lon, lat, 90, radius_m
    )

    lon_s, lat_s, _ = geod.fwd(
        lon, lat, 180, radius_m
    )

    lon_w, lat_w, _ = geod.fwd(
        lon, lat, 270, radius_m
    )

    min_lon = min(lon_w, lon_e)
    max_lon = max(lon_w, lon_e)
    min_lat = min(lat_s, lat_n)
    max_lat = max(lat_s, lat_n)

    x1, y1 = lonlat_to_tile_float(
        min_lon, max_lat, z
    )

    x2, y2 = lonlat_to_tile_float(
        max_lon, min_lat, z
    )

    x_min = int(np.floor(x1))
    x_max = int(np.floor(x2))
    y_min = int(np.floor(y1))
    y_max = int(np.floor(y2))

    tiles = {}

    for x in range(x_min, x_max + 1):

        for y in range(y_min, y_max + 1):

            # 既存のDEM取得関数を使用
            rgb = get_dem_tile(z, x, y)

            if rgb is None:
                continue

            tiles[(x, y)] = rgb_to_elevation(rgb)

    return tiles


# ============================================================
# 都市ごとの仰角計算
# ============================================================

angle_results = []

total_cities = len(gdf_capital_points)

for city_idx, row in enumerate(
    gdf_capital_points.itertuples(index=False),
    start=1
):

    city = row.都市
    office_lon = row.経度
    office_lat = row.緯度

    print(
        f"\n[{city_idx}/{total_cities}] {city}"
    )

    # --------------------------------------------------------
    # 必要なDEMタイルを取得
    # --------------------------------------------------------

    tiles = load_dem_tiles_for_radius(
        office_lon,
        office_lat,
        RADIUS_M,
        z
    )

    print(
        f"  DEMタイル: {len(tiles)}枚"
    )

    if len(tiles) == 0:
        print("  → DEMが取得できませんでした")
        continue

    # --------------------------------------------------------
    # 県庁所在地地点の標高
    # --------------------------------------------------------

    office_elevation = get_elevation_from_tiles(
        np.array([office_lon]),
        np.array([office_lat]),
        tiles,
        z
    )[0]

    if np.isnan(office_elevation):
        print(
            "  → 県庁所在地地点の標高を取得できませんでした"
        )
        continue

    # --------------------------------------------------------
    # 360方向 × 100m～20km
    # --------------------------------------------------------

    azimuth_grid, distance_grid = np.meshgrid(
        AZIMUTHS_DEG,
        DISTANCES_M
    )

    azimuth_flat = azimuth_grid.ravel()
    distance_flat = distance_grid.ravel()

    # 各地点の緯度経度
    lon_flat, lat_flat, _ = geod.fwd(
        np.full(
            len(azimuth_flat),
            office_lon
        ),
        np.full(
            len(azimuth_flat),
            office_lat
        ),
        azimuth_flat,
        distance_flat
    )

    # --------------------------------------------------------
    # DEM標高
    # --------------------------------------------------------

    elevations_flat = get_elevation_from_tiles(
        lon_flat,
        lat_flat,
        tiles,
        z
    )

    # --------------------------------------------------------
    # 仰角
    # --------------------------------------------------------

    elevation_difference = (
        elevations_flat - office_elevation
    )

    angles_rad = np.arctan2(
        elevation_difference,
        distance_flat
    )

    angles_deg = np.degrees(
        angles_rad
    )

    angles_matrix = angles_deg.reshape(
        len(DISTANCES_M),
        len(AZIMUTHS_DEG)
    )

    # 各方向の最大仰角
    max_angles = np.full(
        len(AZIMUTHS_DEG),
        np.nan
    )

    for i in range(
        len(AZIMUTHS_DEG)
    ):

        direction_angles = (
            angles_matrix[:, i]
        )

        valid = ~np.isnan(
            direction_angles
        )

        if valid.any():

            max_angles[i] = np.max(
                direction_angles[valid]
            )

    # --------------------------------------------------------
    # 閉塞方向率
    # --------------------------------------------------------

    valid_directions = ~np.isnan(
        max_angles
    )

    closed_directions = (
        (max_angles > 0)
        & valid_directions
    )

    if valid_directions.sum() > 0:

        closed_ratio = (
            closed_directions.sum()
            / valid_directions.sum()
        )

        mean_max_angle = np.nanmean(
            max_angles
        )

        maximum_angle = np.nanmax(
            max_angles
        )

        angle_results.append({
            "都市": city,
            "県庁所在地標高_m": office_elevation,
            "閉塞方向数": int(
                closed_directions.sum()
            ),
            "有効方向数": int(
                valid_directions.sum()
            ),
            "閉塞方向率_20km": closed_ratio,
            "平均最大仰角_deg": mean_max_angle,
            "最大仰角_deg": maximum_angle
        })

        print(
            f"  → 完了: "
            f"閉塞方向率={closed_ratio:.1%}, "
            f"平均最大仰角={mean_max_angle:.2f}°, "
            f"最大仰角={maximum_angle:.2f}°"
        )

    else:

        print(
            "  → 有効な方向のDEM値を取得できませんでした"
        )


[1/47] 札幌
  DEMタイル: 514枚
  → 完了: 閉塞方向率=74.4%, 平均最大仰角=1.35°, 最大仰角=5.73°

[2/47] 青森
  DEMタイル: 454枚
  → 完了: 閉塞方向率=85.8%, 平均最大仰角=1.33°, 最大仰角=3.85°

[3/47] 盛岡
  DEMタイル: 484枚
  → 完了: 閉塞方向率=100.0%, 平均最大仰角=2.19°, 最大仰角=4.73°

[4/47] 仙台
  DEMタイル: 422枚
  → 完了: 閉塞方向率=74.7%, 平均最大仰角=0.95°, 最大仰角=2.64°

[5/47] 秋田
  DEMタイル: 324枚
  → 完了: 閉塞方向率=100.0%, 平均最大仰角=0.81°, 最大仰角=3.34°

[6/47] 山形
  DEMタイル: 484枚
  → 完了: 閉塞方向率=97.8%, 平均最大仰角=6.65°, 最大仰角=18.44°

[7/47] 福島
  DEMタイル: 484枚
  → 完了: 閉塞方向率=100.0%, 平均最大仰角=3.01°, 最大仰角=5.41°

[8/47] 水戸
  DEMタイル: 389枚
  → 完了: 閉塞方向率=100.0%, 平均最大仰角=0.16°, 最大仰角=1.16°

[9/47] 宇都宮
  DEMタイル: 462枚
  → 完了: 閉塞方向率=82.2%, 平均最大仰角=2.60°, 最大仰角=8.96°

[10/47] 前橋
  DEMタイル: 441枚
  → 完了: 閉塞方向率=78.9%, 平均最大仰角=1.17°, 最大仰角=4.70°

[11/47] さいたま
  DEMタイル: 441枚
  → 完了: 閉塞方向率=100.0%, 平均最大仰角=0.08°, 最大仰角=0.23°

[12/47] 千葉
  DEMタイル: 402枚
  → 完了: 閉塞方向率=100.0%, 平均最大仰角=0.67°, 最大仰角=2.34°

[13/47] 東京
  DEMタイル: 412枚
  → 完了: 閉塞方向率=85.6%, 平均最大仰角=0.23°, 最大仰角=0.64°

[14/47] 横浜
  DEMタイル: 337枚
  → 完了: 閉塞方向率=97.8%, 平均

In [67]:
# ============================================================
# DataFrame化
# ============================================================

# 20kmの結果をDataFrame化
df_elevation_angle_20km = pd.DataFrame(angle_results)

# 20km版としてカラム名を整理
df_elevation_angle_20km = df_elevation_angle_20km.rename(columns={
    "閉塞方向数": "閉塞方向数_20km",
    "有効方向数": "有効方向数_20km",
    "平均最大仰角_deg": "平均最大仰角_20km_deg",
    "最大仰角_deg": "最大仰角_20km_deg"
})

df_elevation_angle_20km

,都市,県庁所在地標高_m,閉塞方向数_20km,有効方向数_20km,閉塞方向率_20km,平均最大仰角_20km_deg,最大仰角_20km_deg
0,札幌,18.14,268,360,0.744444,1.351675,5.727278
1,青森,2.40,309,360,0.858333,1.334971,3.853199
2,盛岡,124.48,360,360,1.000000,2.191747,4.729538
3,仙台,46.51,269,360,0.747222,0.948222,2.635607
4,秋田,5.20,360,360,1.000000,0.807612,3.335346
5,山形,196.68,352,360,0.977778,6.650515,18.444187
6,福島,61.82,360,360,1.000000,3.013704,5.405453
7,水戸,30.00,360,360,1.000000,0.158714,1.163104
8,宇都宮,124.24,296,360,0.822222,2.603707,8.956149
9,前橋,106.99,284,360,0.788889,1.169383,4.696898


### （追加）県庁から半径5km,10km

In [68]:
# ============================================================
# 5km・10kmの閉塞方向率を追加計算
# ============================================================

radius_list_km = [5, 10]

angle_results_by_radius = {}

for radius_km in radius_list_km:

    radius_m = radius_km * 1000

    print("\n" + "=" * 60)
    print(f"半径 {radius_km} km の計算開始")
    print("=" * 60)

    results = []

    total_cities = len(gdf_capital_points)

    for city_idx, row in enumerate(
        gdf_capital_points.itertuples(index=False),
        start=1
    ):
        city = row.都市
        office_lon = row.経度
        office_lat = row.緯度

        print(
            f"\n[{city_idx}/{total_cities}] "
            f"{city}（半径 {radius_km} km）"
        )

        # ----------------------------------------------------
        # DEMタイル取得
        # ----------------------------------------------------
        tiles = load_dem_tiles_for_radius(
            office_lon,
            office_lat,
            radius_m,
            z
        )

        print(f"  DEMタイル: {len(tiles)}枚")

        if len(tiles) == 0:
            print("  → DEMが取得できませんでした")
            continue

        # ----------------------------------------------------
        # 県庁所在地地点の標高
        # ----------------------------------------------------
        office_elevation = get_elevation_from_tiles(
            np.array([office_lon]),
            np.array([office_lat]),
            tiles,
            z
        )[0]

        if np.isnan(office_elevation):
            print(
                "  → 県庁所在地地点の標高を"
                "取得できませんでした"
            )
            continue

        # ----------------------------------------------------
        # 各方向 × 各距離の座標を作成
        # ----------------------------------------------------
        distances_m = np.arange(
            STEP_M,
            radius_m + 1,
            STEP_M
        )

        azimuth_grid, distance_grid = np.meshgrid(
            AZIMUTHS_DEG,
            distances_m
        )

        azimuth_flat = azimuth_grid.ravel()
        distance_flat = distance_grid.ravel()

        # ----------------------------------------------------
        # 緯度経度へ変換
        # ----------------------------------------------------
        lon_flat, lat_flat, _ = geod.fwd(
            np.full(
                len(azimuth_flat),
                office_lon
            ),
            np.full(
                len(azimuth_flat),
                office_lat
            ),
            azimuth_flat,
            distance_flat
        )

        # ----------------------------------------------------
        # DEM標高取得
        # ----------------------------------------------------
        elevations_flat = get_elevation_from_tiles(
            lon_flat,
            lat_flat,
            tiles,
            z
        )

        # ----------------------------------------------------
        # 仰角計算
        # ----------------------------------------------------
        elevation_difference = (
            elevations_flat - office_elevation
        )

        angles_rad = np.arctan2(
            elevation_difference,
            distance_flat
        )

        angles_deg = np.degrees(angles_rad)

        angles_matrix = angles_deg.reshape(
            len(distances_m),
            len(AZIMUTHS_DEG)
        )

        # ----------------------------------------------------
        # 各方向の最大仰角
        # ----------------------------------------------------
        max_angles = np.full(
            len(AZIMUTHS_DEG),
            np.nan
        )

        for i in range(len(AZIMUTHS_DEG)):

            direction_angles = angles_matrix[:, i]

            valid = ~np.isnan(direction_angles)

            if valid.any():
                max_angles[i] = np.max(
                    direction_angles[valid]
                )

        # ----------------------------------------------------
        # 閉塞方向
        # 「県庁所在地より高い地形が存在する方向」
        # ----------------------------------------------------
        valid_directions = ~np.isnan(max_angles)

        closed_directions = (
            (max_angles > 0)
            & valid_directions
        )

        # ----------------------------------------------------
        # 集計
        # ----------------------------------------------------
        if valid_directions.sum() > 0:

            closed_ratio = (
                closed_directions.sum()
                / valid_directions.sum()
            )

            mean_max_angle = np.nanmean(
                max_angles
            )

            maximum_angle = np.nanmax(
                max_angles
            )

            results.append({
                "都市": city,
                f"県庁所在地標高_m_{radius_km}km":
                    office_elevation,
                f"閉塞方向数_{radius_km}km":
                    int(closed_directions.sum()),
                f"有効方向数_{radius_km}km":
                    int(valid_directions.sum()),
                f"閉塞方向率_{radius_km}km":
                    closed_ratio,
                f"平均最大仰角_{radius_km}km_deg":
                    mean_max_angle,
                f"最大仰角_{radius_km}km_deg":
                    maximum_angle
            })

            print(
                f"  → 完了: "
                f"閉塞方向率={closed_ratio:.1%}, "
                f"平均最大仰角="
                f"{mean_max_angle:.2f}°, "
                f"最大仰角="
                f"{maximum_angle:.2f}°"
            )

        else:
            print(
                "  → 有効な方向のDEM値を"
                "取得できませんでした"
            )

    angle_results_by_radius[radius_km] = results


半径 5 km の計算開始

[1/47] 札幌（半径 5 km）
  DEMタイル: 42枚
  → 完了: 閉塞方向率=68.3%, 平均最大仰角=0.82°, 最大仰角=5.31°

[2/47] 青森（半径 5 km）
  DEMタイル: 34枚
  → 完了: 閉塞方向率=73.6%, 平均最大仰角=0.14°, 最大仰角=1.02°

[3/47] 盛岡（半径 5 km）
  DEMタイル: 36枚
  → 完了: 閉塞方向率=100.0%, 平均最大仰角=1.43°, 最大仰角=4.60°

[4/47] 仙台（半径 5 km）
  DEMタイル: 36枚
  → 完了: 閉塞方向率=74.4%, 平均最大仰角=0.89°, 最大仰角=2.64°

[5/47] 秋田（半径 5 km）
  DEMタイル: 33枚
  → 完了: 閉塞方向率=100.0%, 平均最大仰角=0.49°, 最大仰角=1.45°

[6/47] 山形（半径 5 km）
  DEMタイル: 36枚
  → 完了: 閉塞方向率=73.9%, 平均最大仰角=6.09°, 最大仰角=18.44°

[7/47] 福島（半径 5 km）
  DEMタイル: 36枚
  → 完了: 閉塞方向率=100.0%, 平均最大仰角=2.07°, 最大仰角=5.04°

[8/47] 水戸（半径 5 km）
  DEMタイル: 42枚
  → 完了: 閉塞方向率=94.7%, 平均最大仰角=0.03°, 最大仰角=0.19°

[9/47] 宇都宮（半径 5 km）
  DEMタイル: 36枚
  → 完了: 閉塞方向率=82.2%, 平均最大仰角=2.50°, 最大仰角=8.96°

[10/47] 前橋（半径 5 km）
  DEMタイル: 36枚
  → 完了: 閉塞方向率=68.1%, 平均最大仰角=0.22°, 最大仰角=0.67°

[11/47] さいたま（半径 5 km）
  DEMタイル: 36枚
  → 完了: 閉塞方向率=47.8%, 平均最大仰角=0.02°, 最大仰角=0.10°

[12/47] 千葉（半径 5 km）
  DEMタイル: 33枚
  → 完了: 閉塞方向率=100.0%, 平均最大仰角=0.67°, 最大仰角=2.34°

[13/47] 東京（半径

In [69]:
# 5 km・10 kmをDataFrame化

df_elevation_angle_5km = pd.DataFrame(
    angle_results_by_radius[5]
)

df_elevation_angle_10km = pd.DataFrame(
    angle_results_by_radius[10]
)

### 半径5km,10km,20kmの計算結果を結合

In [75]:
# 閉塞方向率（5km,10km,20km）を結合

df_elevation_angle = (
    df_elevation_angle_5km
    .merge(
        df_elevation_angle_10km,
        on="都市",
        how="outer"
    )
    .merge(
        df_elevation_angle_20km,
        on="都市",
        how="outer"
    )
)

#### データの形をきれいにする

In [73]:
# 県庁所在地データの順番を基準にする
capital_order = df_prefectural_capitals["都市"].tolist()

# 閉塞方向率データを都市名で結合
df_elevation_angle_merged = (
    df_prefectural_capitals
    .merge(
        df_elevation_angle,
        on="都市",
        how="left"
    )
)

# 県庁所在地データと同じ順番にする
df_elevation_angle_merged["都市"] = pd.Categorical(
    df_elevation_angle_merged["都市"],
    categories=capital_order,
    ordered=True
)

df_elevation_angle_merged = (
    df_elevation_angle_merged
    .sort_values("都市")
    .reset_index(drop=True)
)

df_elevation_angle_merged.head(2)

,コード,都道府県,県庁所在地,ローマ字,都市,県庁所在地標高_m_5km,閉塞方向数_5km,有効方向数_5km,閉塞方向率_5km,平均最大仰角_5km_deg,...,有効方向数_10km,閉塞方向率_10km,平均最大仰角_10km_deg,最大仰角_10km_deg,県庁所在地標高_m,閉塞方向数_20km,有効方向数_20km,閉塞方向率_20km,平均最大仰角_20km_deg,最大仰角_20km_deg
0,1,北海道,札幌市,sapporo,札幌,18.14,246,360,0.683333,0.816571,...,360,0.697222,1.162107,5.727278,18.14,268,360,0.744444,1.351675,5.727278
1,2,青森県,青森市,aomori,青森,2.40,265,360,0.736111,0.138128,...,360,0.777778,0.686562,2.478329,2.40,309,360,0.858333,1.334971,3.853199


In [79]:
# 同じ数値が入っているカラムを削除
df_elevation_angle_merged = df_elevation_angle_merged.drop(
    columns=[
        "県庁所在地標高_m_5km",
        "県庁所在地標高_m_10km"
    ]
)
df_elevation_angle_merged.head(1)

,コード,都道府県,県庁所在地,ローマ字,都市,閉塞方向数_5km,有効方向数_5km,閉塞方向率_5km,平均最大仰角_5km_deg,最大仰角_5km_deg,...,有効方向数_10km,閉塞方向率_10km,平均最大仰角_10km_deg,最大仰角_10km_deg,県庁所在地標高_m,閉塞方向数_20km,有効方向数_20km,閉塞方向率_20km,平均最大仰角_20km_deg,最大仰角_20km_deg
0,1,北海道,札幌市,sapporo,札幌,246,360,0.683333,0.816571,5.312997,...,360,0.697222,1.162107,5.727278,18.14,268,360,0.744444,1.351675,5.727278


In [89]:
df_elevation_angle_merged.columns

Index(['コード', '都道府県', '県庁所在地', 'ローマ字', '都市', '閉塞方向数_5km', '有効方向数_5km',
       '閉塞方向率_5km', '平均最大仰角_5km_deg', '最大仰角_5km_deg', '閉塞方向数_10km',
       '有効方向数_10km', '閉塞方向率_10km', '平均最大仰角_10km_deg', '最大仰角_10km_deg',
       '県庁所在地標高_m', '閉塞方向数_20km', '有効方向数_20km', '閉塞方向率_20km',
       '平均最大仰角_20km_deg', '最大仰角_20km_deg'],
      dtype='str')

In [85]:
# 閉塞方向率データを都市名で結合
df_analysis_bonchi_elevation = df_prefectural_capitals.merge(
    df_elevation,
    on="県庁所在地",
    how="left"
)
df_analysis_bonchi_elevation.head(2)

,コード,都道府県,県庁所在地,ローマ字,都市,最低標高_m,最高標高_m,市域内標高差_m,市域内平均標高_m,市域内標高標準偏差_m
0,1,北海道,札幌市,sapporo,札幌,1.00,1486.63,1485.63,420.611535,348.341252
1,2,青森県,青森市,aomori,青森,-4.22,1583.62,1587.84,275.008913,296.808635


In [86]:
# csv出力
### 標高差と標準偏差
df_analysis_bonchi_elevation.to_csv("../data/df_analysis_bonchi_elevation.csv")
### 県庁から360度の仰角
df_elevation_angle_merged.to_csv("../data/df_analysis_bonchi_angle.csv")